In [14]:
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, DistilBertForSequenceClassification
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 一 配置

In [7]:
class Arguments(object):
    train_data_path = 'data_train.jsonl'
    test_data_path = 'data_test.jsonl'

    train_len = 0
    test_len = 0
    with open(train_data_path,'r')as file:
        for line in file:
            # 解析每一行的json数据
            data = json.loads(line)
            text_len = len(data['text'])
            if text_len > train_len:
                train_len = text_len
    with open(test_data_path,'r')as file:
        for line in file:
            # 解析每一行的json数据
            data = json.loads(line)
            text_len = len(data['text'])
            if text_len > test_len:
                test_len = text_len

    print('train_len:',train_len)
    print('test_len:',test_len)

    bert_base_path = '/Users/bowie/Documents/muti-model/bert-base-uncased'
    bert_distil_path = '/Users/bowie/Documents/muti-model/distilbert-base-uncased'
    epoch = 10
    max_len = 360
    lr=5e-5


args = Arguments()


train_len: 356
test_len: 298


In [8]:
# 1. 直接加载文件，不指定 split，这样只会加载整个文件而不会尝试分割数据
# train_data = load_dataset('json', data_files=args.train_data_path, split='train')

# 2. 手动加载多个分割
data_files = {
    'train': args.train_data_path,
    'test': args.test_data_path
}
datasets = load_dataset('json', data_files=data_files)

# # 加载 'test' 分割
# test_data = datasets['test']

# 3. 检查单一文件并手动划分 如果你的数据都在一个文件中，你需要手动从数据中分出 test 集：
# 加载整个数据集
# datasets = load_dataset('json', data_files=args.train_data_path)

# 手动划分 test 数据 (例如前 20% 数据作为测试集)
# split_ratio = 0.2
# split_data = datasets['train'].train_test_split(test_size=split_ratio)

# DatasetDict({
#     train: Dataset({
#         features: ['text', 'label'],
#         num_rows: 12800
#     })
#     test: Dataset({
#         features: ['text', 'label'],
#         num_rows: 3200
#     })
# })


In [9]:
datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [10]:
tokenizer = BertTokenizer.from_pretrained(args.bert_base_path)

/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [11]:
def collote_fn(batch_samples):
    batch_text= []
    batch_label = []
    for sample in batch_samples:
        batch_text.append(sample['text'])
        batch_label.append(int(sample['label']))
    X = tokenizer(
        batch_text,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    y = torch.tensor(batch_label)
    return X, y


In [12]:
train_dataloader = DataLoader(datasets['train'], batch_size=4, shuffle=True, collate_fn=collote_fn)
test_dataloader = DataLoader(datasets['test'], batch_size=4, collate_fn=collote_fn)

In [30]:
for i in train_dataloader:
    print(i)
    break

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


({'input_ids': tensor([[  101,  1045,  2074,  2031,  1037,  3110,  2003,  2183,  2000,  6293,
          2007,  2014,  1998, 10047,  8295,  2009,   102,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101, 10047,  3110, 12422,  1998, 14477, 21408,  8737, 13602,  2098,
          2035,  2058,  2153,   102,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  4921,  2063,  2371,  3294, 13394,  2021,  2005,  1996,  2087,
          2112, 10047,  3110, 21931,  1998, 18836,  2005,  1996,  3815,  1997,
          2490,  1045,  2031,  2013,  2026,  2155,  1998,  2814,   102],
        [  101,  1045,  2113,  2054,  2003,  3308,  1045,  2514,  2061, 17380,
          2000,  2514,  2488,   102,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [46]:
# 加载BERT模型
teacher_model = BertForSequenceClassification.from_pretrained(args.bert_base_path, num_labels=6)

A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Pl

In [47]:
student_model = DistilBertForSequenceClassification.from_pretrained(args.bert_distil_path, num_labels=6)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at /Users/bowie/Documents/muti-model/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# state_dict = torch.load('teacher_model50.pt')
# teacher_model.load_state_dict(state_dict)

In [48]:
teacher_model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [49]:
# 将教师模型设置为评估模式
teacher_model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [50]:
student_model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [51]:
# 定义优化器
optimizer = AdamW(student_model.parameters(), lr=5e-5)

/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [52]:
# 设置温度和 alpha 值
temperature = 2.0
alpha = 0.5

In [ ]:
# import torch
# import torch.nn.functional as F

# def distillation_loss(student_logits, teacher_logits, labels, temperature=2.0, alpha=0.5):
#     """
#     蒸馏损失函数
#     - student_logits: 学生模型的输出 logits
#     - teacher_logits: 教师模型的输出 logits
#     - labels: 真实标签
#     - temperature: 温度参数
#     - alpha: 交叉熵损失和KL散度损失的权重系数
#     """
#     # 学生模型与真实标签的交叉熵损失
#     student_loss = F.cross_entropy(student_logits, labels)
    
#     # 温度调节，平滑logits
#     teacher_logits = teacher_logits / temperature
#     student_logits = student_logits / temperature
    
#     # KL散度损失：学生模型与教师模型的logits之间的距离
#     distillation_loss = F.kl_div(F.log_softmax(student_logits, dim=-1), 
#                                  F.softmax(teacher_logits, dim=-1), 
#                                  reduction='batchmean') * (temperature ** 2)
    
#     # 总损失：结合交叉熵损失和蒸馏损失
#     total_loss = alpha * student_loss + (1 - alpha) * distillation_loss
#     return total_loss


In [53]:

def distillation_loss(student_logits, teacher_logits, labels, temperature, alpha):
    """
    蒸馏损失函数
    student_logits: 学生模型的logits
    teacher_logits: 教师模型的logits
    labels: 真实标签
    temperature: 温度参数，用于平滑教师模型输出
    alpha: 损失函数中交叉熵损失和KL散度的加权系数
    """
    # 学生模型和真实标签的交叉熵损失
    student_loss = F.cross_entropy(student_logits, labels)
    
    # 将教师模型的logits按温度进行平滑
    teacher_logits = teacher_logits / temperature
    student_logits = student_logits / temperature
    
    # 学生模型和教师模型之间的KL散度
    distillation_loss = F.kl_div(F.log_softmax(student_logits, dim=-1), 
                                 F.softmax(teacher_logits, dim=-1), 
                                 reduction='batchmean')
    
    # 结合两种损失
    loss = alpha * student_loss + (1 - alpha) * distillation_loss
    return loss


In [55]:
total_step = len(train_dataloader) * 1
global_step = 1

# 训练循环
for epoch in range(1):
    student_model.train()
    
    for batch, y in train_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = y.to(device)
        
        # 禁用教师模型的梯度
        with torch.no_grad():
            teacher_outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
            teacher_logits = teacher_outputs.logits
        
        # 学生模型的输出
        student_outputs = student_model(input_ids=input_ids, attention_mask=attention_mask)
        student_logits = student_outputs.logits
        
        # 计算蒸馏损失
        loss = distillation_loss(student_logits, teacher_logits, labels, temperature=temperature, alpha=alpha)
        
        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f'Epoch: {epoch}, {global_step}/{total_step}, Loss: {loss.item()}')
        global_step += 1

Epoch: 0,1/4000, Loss: 0.27542227506637573
Epoch: 0,2/4000, Loss: 0.43258869647979736
Epoch: 0,3/4000, Loss: 0.36064454913139343
Epoch: 0,4/4000, Loss: 0.24570529162883759
Epoch: 0,5/4000, Loss: 0.2263937145471573
Epoch: 0,6/4000, Loss: 0.26603925228118896
Epoch: 0,7/4000, Loss: 0.25241705775260925
Epoch: 0,8/4000, Loss: 0.6061501502990723
Epoch: 0,9/4000, Loss: 0.6103531122207642
Epoch: 0,10/4000, Loss: 0.22693264484405518
Epoch: 0,11/4000, Loss: 0.22512897849082947
Epoch: 0,12/4000, Loss: 0.24796143174171448
Epoch: 0,13/4000, Loss: 0.23631267249584198
Epoch: 0,14/4000, Loss: 0.2838065028190613
Epoch: 0,15/4000, Loss: 0.25520750880241394
Epoch: 0,16/4000, Loss: 0.30076056718826294
Epoch: 0,17/4000, Loss: 0.21999268233776093
Epoch: 0,18/4000, Loss: 0.35566920042037964
Epoch: 0,19/4000, Loss: 0.8638248443603516
Epoch: 0,20/4000, Loss: 0.2461794912815094
Epoch: 0,21/4000, Loss: 0.21837882697582245
Epoch: 0,22/4000, Loss: 0.21692490577697754
Epoch: 0,23/4000, Loss: 0.6167454123497009
Epoc

## 保存学生模型 

In [1]:
# 假设训练完成后，选择一个保存路径
save_directory = './distilled_student_model'

In [56]:
# 保存学生模型
student_model.save_pretrained(save_directory)

# 保存 tokenizer（如果需要的话）
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
tokenizer.save_pretrained(save_directory)

/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


('./distilled_student_model/tokenizer_config.json',
 './distilled_student_model/special_tokens_map.json',
 './distilled_student_model/vocab.txt',
 './distilled_student_model/added_tokens.json')

In [2]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

# 加载保存的学生模型
student_model = DistilBertForSequenceClassification.from_pretrained(save_directory)

# 加载 tokenizer
tokenizer = DistilBertTokenizer.from_pretrained(save_directory)


/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch

# 准备测试数据（假设 test_sentences 是你的测试句子列表）
test_sentences = ["im really feeling helpless", "i feel the cold is what i m trying to say"]  # 示例

# 对测试数据进行编码
inputs = tokenizer(test_sentences, padding=True, truncation=True, return_tensors='pt')

# 模型推理
student_model.eval()  # 设置为评估模式
with torch.no_grad():
    outputs = student_model(**inputs)
    logits = outputs.logits

# 获取预测标签
predictions = torch.argmax(logits, dim=-1)
predictions

tensor([4, 3])

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 假设 y_true 是真实标签，y_pred 是预测标签
y_true = [...]  # 真实标签
y_pred = predictions.numpy()  # 将预测标签转为numpy数组

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

print(f'准确率: {accuracy:.2f}')
print(f'精确率: {precision:.2f}')
print(f'召回率: {recall:.2f}')
print(f'F1-score: {f1:.2f}')


In [18]:
student_model.eval()
all_pred = []
all_true = []

val_step = 1

with torch.no_grad():
    for batch, y in test_dataloader:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']

        # 模型预测
        outputs = student_model(input_ids, attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

        all_pred.extend(predictions)
        all_true.extend(y.numpy())

        val_step += 1
        print(val_step)

# 评估意图分类
accuracy = accuracy_score(all_true, all_pred)
precision, recall, f1, _ = precision_recall_fscore_support(all_true, all_pred, average='macro')
print(f'准确率: {accuracy:.2f}')
print(f'精确率: {precision:.2f}')
print(f'召回率: {recall:.2f}')
print(f'F1-score: {f1:.2f}')

2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
27